# Final ResNet34 Training

## Objective

After selecting ResNet34 as the best-performing architecture based on Stratified 5-Fold Cross Validation, the final model is retrained using the complete dataset.

Unlike the previous experiments, no validation split is used during this stage. Instead, every available image is utilized for training to maximize the amount of data seen by the model before deployment.

The trained weights produced in this notebook are used by both the command-line inference script (`predict.py`) and the Streamlit web application (`app.py`).

## Import Required Libraries

Import all libraries required for transfer learning, optimization, image preprocessing, and model training.

In [1]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cpu


## Define Image Preprocessing Pipeline

Create the image preprocessing and augmentation pipeline used during final model training.

The images are resized, augmented, converted into tensors, and normalized using ImageNet statistics.

In [3]:
train_transform = transforms.Compose([

    transforms.Resize((256,256)),

    transforms.RandomCrop(224),

    transforms.RandomHorizontalFlip(0.5),

    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

In [4]:
dataset = datasets.ImageFolder(
    "../dataset",
    transform=train_transform
)

train_loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True
)

print(len(dataset))

102


## Initialize the ResNet34 Model

Load the ImageNet pretrained ResNet34 architecture.

Replace the original classification layer with a two-class classifier suitable for distinguishing between real photographs and screen recaptures.

In [5]:
weights = models.ResNet34_Weights.DEFAULT

model = models.resnet34(
    weights=weights
)

# Freeze everything
for param in model.parameters():
    param.requires_grad = False

# Fine-tune Layer 3
for param in model.layer3.parameters():
    param.requires_grad = True

# Fine-tune Layer 4
for param in model.layer4.parameters():
    param.requires_grad = True

# Replace classifier
model.fc = nn.Linear(512,2)

model = model.to(device)

In [6]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

optimizer = optim.AdamW(

    filter(
        lambda p:p.requires_grad,
        model.parameters()
    ),

    lr=5e-5,

    weight_decay=1e-4

)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="min",

    factor=0.5,

    patience=3

)

## Define Training Function

Implement the training loop responsible for performing forward propagation, backpropagation, and parameter updates during each training epoch.

In [7]:
def train_one_epoch():

    model.train()

    running_loss = 0

    correct = 0

    total = 0

    for images,labels in train_loader:

        images = images.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _,preds = torch.max(outputs,1)

        total += labels.size(0)

        correct += (preds==labels).sum().item()

    loss = running_loss/len(train_loader)

    acc = correct/total

    return loss,acc

## Train the Final Model

Train the ResNet34 model using the complete dataset.

The model achieving the lowest training loss is stored as the final deployment checkpoint.

In [8]:
best_loss = 999

best_model = copy.deepcopy(
    model.state_dict()
)

counter = 0

PATIENCE = 5

EPOCHS = 40

for epoch in range(EPOCHS):

    loss,acc = train_one_epoch()

    scheduler.step(loss)

    print(

        f"Epoch {epoch+1:02d} | "

        f"Loss: {loss:.4f} | "

        f"Accuracy: {acc:.4f}"

    )

    if loss < best_loss:

        best_loss = loss

        best_model = copy.deepcopy(
            model.state_dict()
        )

        counter = 0

    else:

        counter += 1

    if counter >= PATIENCE:

        print("Early Stopping")

        break

Epoch 01 | Loss: 0.8435 | Accuracy: 0.5294
Epoch 02 | Loss: 0.4591 | Accuracy: 0.8529
Epoch 03 | Loss: 0.2908 | Accuracy: 0.9804
Epoch 04 | Loss: 0.2787 | Accuracy: 0.9706
Epoch 05 | Loss: 0.2711 | Accuracy: 0.9608
Epoch 06 | Loss: 0.2944 | Accuracy: 0.9412
Epoch 07 | Loss: 0.2559 | Accuracy: 0.9804
Epoch 08 | Loss: 0.2550 | Accuracy: 0.9706
Epoch 09 | Loss: 0.2431 | Accuracy: 1.0000
Epoch 10 | Loss: 0.2529 | Accuracy: 0.9804
Epoch 11 | Loss: 0.2410 | Accuracy: 1.0000
Epoch 12 | Loss: 0.2384 | Accuracy: 1.0000
Epoch 13 | Loss: 0.2267 | Accuracy: 1.0000
Epoch 14 | Loss: 0.2575 | Accuracy: 0.9902
Epoch 15 | Loss: 0.2263 | Accuracy: 1.0000
Epoch 16 | Loss: 0.2448 | Accuracy: 0.9902
Epoch 17 | Loss: 0.2460 | Accuracy: 0.9902
Epoch 18 | Loss: 0.2954 | Accuracy: 0.9314
Epoch 19 | Loss: 0.2356 | Accuracy: 0.9902
Epoch 20 | Loss: 0.2435 | Accuracy: 0.9902
Early Stopping


In [9]:
model.load_state_dict(best_model)

torch.save(

    model.state_dict(),

    "../models/final_resnet34_detector.pth"

)

print("Final model saved!")

Final model saved!


In [10]:
model.eval()

correct = 0

total = 0

with torch.no_grad():

    for images,labels in train_loader:

        images = images.to(device)

        labels = labels.to(device)

        outputs = model(images)

        _,preds = torch.max(outputs,1)

        total += labels.size(0)

        correct += (preds==labels).sum().item()

print(f"Training Accuracy : {correct/total:.4f}")

Training Accuracy : 1.0000


# Summary

This notebook retrained the selected ResNet34 architecture using the complete dataset after model selection.

### Training Strategy

- ImageNet pretrained ResNet34
- Complete dataset training
- Data augmentation
- AdamW optimizer
- Learning-rate scheduling

The resulting model serves as the final deployment model for this project and is used during inference through both the command-line interface and the Streamlit web application.

Final model:

```text
models/final_resnet34_detector.pth
```